<div style="font-family: system-ui, -apple-system, sans-serif; text-align: center; padding: 48px 24px 24px;">
    <div style="display: inline-block; background: #be0f05; color: white;
                padding: 12px 20px; border-radius: 12px; margin-bottom: 20px;">
        <span style="font-size: 34px; font-weight: 800; letter-spacing: -0.5px;">TIP4PATLIBS &ndash; What is a patent worth? The model</span>
    </div>
    <div style="font-size: 16px; color: #475569; margin-bottom: 8px; line-height: 1.6;">
        Inside <strong>EPO&nbsp;IPScore</strong>: forty questions, a score &mdash; and the eight answers that actually decide the money. Worked through on one real patent, <strong>EP3074539B1</strong> (Q-Linea&nbsp;AB).
    </div>
    <div style="font-size: 13px; color: #94a3b8; margin-bottom: 32px;">
        EPO Academy Training Material &nbsp;&middot;&nbsp; <a href="https://patentreports.depa.tech" target="_blank"
           style="color: #be0f05; text-decoration: none; font-weight: 600;">created by Arne Kr&uuml;ger</a>
        &nbsp;&middot;&nbsp; model: <strong>EPO IPScore 3.01</strong>
        &nbsp;&middot;&nbsp; scenario analysis after Riccardo Priore's NPV Target Planner
    </div>
    <div style="background: #f8fafc; border-radius: 12px; padding: 24px 28px; max-width: 680px;
                margin: 0 auto; border: 1px solid #e2e8f0; text-align: left;">
        <div style="font-size: 14px; color: #334155; line-height: 1.9;">
            <strong>What this notebook does</strong>
            <br/>Step&nbsp;1 &nbsp;&middot;&nbsp; The questionnaire: 40 questions in 5 sections
            <br/>Step&nbsp;2 &nbsp;&middot;&nbsp; <strong>The surprise: only 8 of them carry money</strong>
            <br/>Step&nbsp;3 &nbsp;&middot;&nbsp; The first read-out: points, risk and opportunity
            <br/>Step&nbsp;4 &nbsp;&middot;&nbsp; The second read-out: seven figures from the accounts
            <br/>Step&nbsp;5 &nbsp;&middot;&nbsp; The ten-year cash flow, line by line
            <br/>Step&nbsp;6 &nbsp;&middot;&nbsp; Two things everybody gets wrong
            <br/>Step&nbsp;7 &nbsp;&middot;&nbsp; <strong>The acceptance test</strong> &mdash; does our engine equal the EPO's?
        </div>
    </div>
    <div style="background: #fdf2f2; border-radius: 10px; padding: 16px 24px; max-width: 680px;
                margin: 28px auto 0; border: 1px solid #fecaca;">
        <div style="font-size: 14px; color: #404955; font-weight: 600;">&#9654; &nbsp;Part 1 of 4 &mdash; and the only one you can run anywhere.</div>
        <div style="font-size: 12px; color: #64748b; margin-top: 6px; line-height: 1.6;">
            No PATSTAT, no database, no internet: everything here is arithmetic over a small
            specification file. Notebook&nbsp;2 is the one that needs <strong>EPO&nbsp;TIP</strong>.
        </div>
    </div>
</div>

---

## The idea in one paragraph

A patent is not worth anything by itself. It is worth what it lets a company earn — or stop
losing — over the years it is still in force. **IPScore**, developed by the EPO, turns that
sentence into a procedure: answer 40 structured questions about the patent on a 1–5 scale, and
the model gives you two things. A **profile** — where this patent is strong and where it is
exposed — and a **number**: the Net Present Value of the technology it protects.

This notebook opens the machine and shows every wheel. We build the engine ourselves, in
Python, and at the end we check it against the EPO's own workbook. If our number differs from
theirs, nothing else in this module is worth reading — so that check comes before anything else
we do with it.

---

## Step 1 — The questionnaire: 40 questions, 5 sections

Everything the model knows about a patent arrives through 40 questions. They are grouped into
five sections, each looking at the patent from one angle:

| | Section | Asks about |
|---|---|---|
| **A** | Legal status | Is it granted, how long does it run, how broad, where does it apply |
| **B** | Technology | Is it new, is it better, can it be worked, how does it fit the company |
| **C** | Market conditions | Is there a market, is it growing, who else is in it |
| **D** | Finance | What does it cost to develop, produce, and equip |
| **E** | Strategy | Does this patent serve what the company is actually trying to do |

Each question is answered on a **1–5 scale**, where 1 is the weakest and 5 the strongest
position. 40 questions × 5 points = **200 points maximum**.

The cell below loads that question set. It comes from `ipscore_spec.json` — a specification
extracted once from the EPO's IPScore 3.01 workbook, so that this notebook needs nothing but
itself.

In [1]:
import pandas as pd
import ipscore_kit as kit

spec = kit.load_spec()

overview = pd.DataFrame([
    {
        "Section": f"{key} · {title}",
        "Questions": len(spec.of_section(key)),
        "Max points": 5 * len(spec.of_section(key)),
        "Carry money": sum(1 for q in spec.of_section(key) if q.carries_money),
    }
    for key, title in spec.sections.items()
])
overview.loc["Σ"] = ["Total", len(spec.questions), spec.max_points,
                     len(spec.oek_questions)]
overview

,Section,Questions,Max points,Carry money
0,A · Legal status,8,40,0
1,B · Technology,9,45,1
2,C · Market conditions,9,45,3
3,D · Finance,6,30,4
4,E · Strategy,8,40,0
Σ,Total,40,200,8


Have a look at what a single question actually contains. Besides the question itself, the model
carries the **five answer options** (so two people scoring the same patent mean the same thing
by "4"), an explanation, and a short label for the factor being judged.

In [2]:
a1 = spec["A1"]
print(f"{a1.id} — {a1.factor}")
print(f"\n{a1.question}\n")
for score, answer in enumerate(a1.answers, start=1):
    print(f"   {score}  {answer}")
print(f"\nWhy it is asked: {a1.explanation[:300]}…")

A1 — Patent status

What is the status of the patent?

   1  Patent not yet applied for
   2  Patent application filed
   3  Novelty search and patentability evaluation completed
   4  Patent granted
   5  Opposition period expired

Why it is asked: A patent application involves a considerable degree of uncertainty, in terms of whether the patent will be granted after novelty searching, evaluation of inventive step, and so forth. Hence a patent that has been granted gets a higher score than a patent application that has only recently been filed…


---

## Step 2 — The surprise: only 8 of the 40 questions carry money

This is the single most important thing to understand about IPScore, and the least obvious.

Of the 40 questions, **exactly 8 feed the financial model**. In the EPO workbook they are
marked as *OEK* questions — economic-model questions. Each of them maps a 1–5 answer onto a
concrete economic quantity: an answer of "3" to the question about market growth does not mean
"3 points", it means **5 % growth per year**.

The other **32 questions never touch the Net Present Value at all**. They shape the profile,
the radar chart, the risk and opportunity picture — but you can change all 32 of them from 1 to
5 and the number at the bottom does not move by a cent.

That is not a flaw to hide; it is a fact to state out loud in front of a client. It tells you
exactly where a discussion about the valuation can be productive, and where it cannot.

In [3]:
money = pd.DataFrame([
    {
        "Q": q.id,
        "Factor": q.factor,
        "Economic parameter": q.oek_param.replace("_", " "),
        "Answer 1": q.oek_values[0],
        "Answer 2": q.oek_values[1],
        "Answer 3": q.oek_values[2],
        "Answer 4": q.oek_values[3],
        "Answer 5": q.oek_values[4],
    }
    for q in spec.oek_questions
])
money

,Q,Factor,Economic parameter,Answer 1,Answer 2,Answer 3,Answer 4,Answer 5
0,B5,Pre-commercial term of development,years to market,5.000,2.000,1.00,0.500,0.000
1,C2,Market growth rate,market growth,0.005,0.025,0.05,0.080,0.150
2,C3,Life expectancy,life expectancy,0.500,1.000,2.00,4.000,8.000
3,C6,Potential extra turnover,extra turnover share,0.005,0.020,0.04,0.060,0.100
4,D1,Business output maintainability,output maintainable,1.000,0.750,0.50,0.250,0.000
5,D2,Future cost of development,development cost share,0.300,0.150,0.08,0.025,0.005
6,D3,Cost of production,production cost index,1.300,1.150,1.00,0.850,0.700
7,D4,Investment intensity,investment index,1.200,1.100,1.00,0.700,0.500


Read one row to see how the bridge works. **C2 — market growth**: a score of 1 means the model
uses 0.5 % annual growth, a score of 5 means 15 %. Everything in between is fixed by the EPO's
table, not by us. The same holds for the years before market entry (B5), how long the
technology stays relevant (C3), how much extra turnover it brings (C6), and the four finance
questions D1–D4.

The chart below is the whole questionnaire at a glance: 40 dots, one per question, grouped by
section. The eight filled ones are the only ones that reach the money.

In [4]:
import plotly.graph_objects as go

sections = list(spec.sections)
rows = []
for q in spec.questions:
    rows.append({
        "x": int(q.id[1:]),
        "y": len(sections) - sections.index(q.section),
        "id": q.id,
        "factor": q.factor,
        "money": q.carries_money,
    })
points = pd.DataFrame(rows)

fig = go.Figure()
for carries, label, color in [
    (False, "Profile only — never reaches the NPV", kit.PALETTE["inactive"]),
    (True, "Carries money — feeds the cash flow", kit.PALETTE["revenue"]),
]:
    part = points[points["money"] == carries]
    fig.add_trace(go.Scatter(
        x=part["x"], y=part["y"], mode="markers+text", name=label,
        marker={"size": 38, "color": color,
                "line": {"width": 2, "color": kit.PALETTE["surface"]}},
        text=part["id"], textposition="middle center",
        textfont={"size": 11, "color": "white" if carries else kit.PALETTE["ink_secondary"]},
        customdata=part["factor"],
        hovertemplate="<b>%{text}</b><br>%{customdata}<extra></extra>",
    ))

fig.update_layout(
    **kit.CHART_LAYOUT,
    height=420,
    title="Only 8 of the IPScore questions reach the Net Present Value",
)
fig.update_xaxes(visible=False, range=[0.3, 9.7])
fig.update_yaxes(
    tickmode="array",
    tickvals=[len(sections) - i for i in range(len(sections))],
    ticktext=[f"{k} · {v}" for k, v in spec.sections.items()],
    range=[0.4, len(sections) + 0.6], showgrid=False,
)
fig.show()

---

## Step 3 — The first read-out: points, risk and opportunity

Now we answer the questionnaire — for a **real patent**: `EP3074539B1`, *"Method for detecting
and characterising a microorganism"*, held by **Q-Linea AB** of Uppsala. It covers rapid
identification and antibiotic-susceptibility testing of bacteria, and it was picked out of
module 5's antibiotic-resistance corpus, so the two modules describe the same field: module 5
maps where antimicrobial resistance research stands, module 6 values one patent inside it.

Below is a complete set of 40 answers for it. They are **an adviser's first pass** — what a
person writes down in a first session with a client, before checking anything. Notebook 2 goes
and checks.

In [5]:
from ipscore_kit import Answer

# The forty answers for EP3074539B1 (Q-Linea AB) - written out by hand, because that is
# exactly what filling in IPScore is: a person deciding forty things. This is the
# ADVISER'S FIRST PASS, before anything has been checked against a database, so every
# answer is `judgement`. Notebook 2 measures what PATSTAT can decide and corrects it;
# the difference between the two is what module 6 exists to show.
scores = {
    # A - Legal status
    "A1": 4, "A2": 4, "A3": 4, "A4": 3, "A5": 3, "A6": 2, "A7": 3, "A8": 3,
    # B - Technology
    "B1": 4, "B2": 4, "B3": 3, "B4": 4,
    "B5": 3,                              # money: 1 year before market entry
    "B6": 3, "B7": 4, "B8": 3, "B9": 3,
    # C - Market conditions
    "C1": 4,
    "C2": 4,                              # money: 8 % market growth
    "C3": 5,                              # money: 8 years of life expectancy
    "C4": 3, "C5": 3,
    "C6": 3,                              # money: 4 % extra turnover
    "C7": 3, "C8": 3, "C9": 4,
    # D - Finance
    "D1": 4,                              # money: 25 % of output holds without the patent
    "D2": 3,                              # money: 8 % development cost
    "D3": 4,                              # money: production cost down to 0.85
    "D4": 4,                              # money: investment intensity down to 0.7
    "D5": 3, "D6": 3,
    # E - Strategy
    "E1": 4, "E2": 3, "E3": 4, "E4": 3, "E5": 3, "E6": 4, "E7": 4, "E8": 3,
}

# Written out here for teaching, but the module keeps one copy of the truth in
# worked_example.json. This assert is what stops the two drifting apart.
assert scores == kit.load_worked_example()["scores"], (
    "notebook 1 and worked_example.json disagree - change one, change the other")

answers = {qid: Answer(score) for qid, score in scores.items()}

print(f"{len(answers)} of {len(spec.questions)} questions answered.")
print(f"Provenance: all {len(answers)} are 'judgement' - nothing here has been checked "
      f"against anything.")

40 of 40 questions answered.
Provenance: all 40 are 'judgement' - nothing here has been checked against anything.


Every one of those forty answers carries a **provenance marker**, and right now all forty say
`judgement`. That is module 6's one addition to the EPO model, and it is worth being precise
about what the three markers mean:

* **`measured`** — a PATSTAT query decided this. Nobody's opinion is involved.
* **`informed`** — data narrowed the choice, but a person still picked the level.
* **`judgement`** — expert opinion, and nothing else.

IPScore as the EPO ships it has no such marker, because it does not need one: **all forty
answers are always judgement**. That is the honest description of every IPScore valuation in
existence, including the one in the IPScore reference.

`kit.PATSTAT_CANDIDATES` lists the eleven questions a PATSTAT query can speak to — four of them
strongly — and notebook 2 is where they stop being opinions. Until it has run, a valuation
built on this answer set is a *structured opinion*, and the report says so in large type.

From those answers the model reads off the **profile**. Two numbers per section — the points
scored out of the maximum — plus two averages that drive the risk/opportunity matrix:

* a question flagged as a **risk driver** contributes `(5 − score) × −0.25`: nothing at the
  best answer, −1 at the worst. It measures *how much worse than perfect* this aspect is.
* a question flagged as an **opportunity driver** contributes `(score − 1) × 0.25`: 0 at the
  worst answer, +1 at the best. It measures *how much upside* this aspect offers.

Both are averaged over the flagged questions only — 21 carry a risk flag, 15 an opportunity
flag, and several carry both.

In [6]:
profile = kit.profile(answers, spec)

per_section = pd.DataFrame([
    {
        "Section": f"{key} · {title}",
        "Points": profile.section_points[key],
        "of": 5 * len(spec.of_section(key)),
        "%": round(100 * profile.section_points[key] / (5 * len(spec.of_section(key)))),
    }
    for key, title in spec.sections.items()
])
display(per_section)

print(f"Total score          {profile.total_points} / {profile.max_points}"
      f"  ({100 * profile.total_points / profile.max_points:.0f} %)")
print(f"Average risk         {profile.average_risk:+.2f}   (0 = no exposure, −1 = worst case)")
print(f"Average opportunity  {profile.average_opportunity:+.2f}   (0 = no upside, +1 = best case)")
print(f"Answer provenance    {profile.provenance_counts}")

,Section,Points,of,%
0,A · Legal status,26,40,65
1,B · Technology,31,45,69
2,C · Market conditions,32,45,71
3,D · Finance,21,30,70
4,E · Strategy,28,40,70


Total score          138 / 200  (69 %)
Average risk         -0.39   (0 = no exposure, −1 = worst case)
Average opportunity  +0.63   (0 = no upside, +1 = best case)
Answer provenance    {'measured': 0, 'informed': 0, 'judgement': 40}


---

## Step 4 — The second read-out: seven figures from the accounts

The score alone cannot produce a Euro amount — it has no idea how big the company is. So the
financial half of the model takes **seven figures straight from the annual accounts**:

| Figure | What it is |
|---|---|
| Business turnover | total annual turnover |
| Direct costs | material and production costs |
| Indirect costs | overheads |
| Provision for depreciation | annual depreciation |
| Depreciation period | over how many years equipment is written off |
| Share of current turnover | how much of the company the affected business area is |
| Discount factor | the interest rate future money is discounted at |

From these the model derives three ratios it actually calculates with: the **cost share**
(what fraction of turnover is eaten by costs), the **depreciation share**, and an
**investment index** — how capital-intensive this business is. Note what is *not* in the list:
nothing about the patent. The accounts describe the company; the eight OEK answers describe
what the patent changes about it.

The figures below are the same company the EPO uses in the first of its own test patents, so
that the acceptance test in step 7 lands on familiar ground.

In [7]:
# The same figures notebook 4 uses, from the module's one example file.
example = kit.load_worked_example()
company = example["financials"]

print("!! " + example["financials_note"] + "\n")
print(f"Turnover             {company.turnover:,.0f} EUR")
print(f"Cost share           {company.cost_share:.0%}  of turnover goes to direct + indirect costs")
print(f"Depreciation share   {company.depreciation_share_pct:.1f} %")
print(f"Investment index     {company.investment_index:.2f}   (depreciation period x depreciation share)")

!! Illustrative figures for a mid-size diagnostics company - NOT Q-Linea AB's accounts. PATSTAT holds no financial data; these were chosen to make the model legible and must not be read as anything about the real company.

Turnover             1,500,000 EUR
Cost share           75%  of turnover goes to direct + indirect costs
Depreciation share   3.0 %
Investment index     0.21   (depreciation period x depreciation share)


And here is the bridge in action: the eight money-carrying answers, translated into the
economic parameters the cash flow runs on.

In [8]:
oek = kit.oek_from_answers(answers, spec)

bridge = pd.DataFrame([
    {
        "Q": q.id,
        "Answer given": answers[q.id].score,
        "In words": q.answers[answers[q.id].score - 1],
        "Parameter": q.oek_param.replace("_", " "),
        "Value used": oek[q.oek_param],
    }
    for q in spec.oek_questions
])
bridge

,Q,Answer given,In words,Parameter,Value used
0,B5,3,1 year,years to market,1.00
1,C2,4,High (8%),market growth,0.08
2,C3,5,8 years,life expectancy,8.00
3,C6,3,Medium (4%),extra turnover share,0.04
4,D1,4,25%,output maintainable,0.25
5,D2,3,High (8%),development cost share,0.08
6,D3,4,15% decrease due to use of the patented techno...,production cost index,0.85
7,D4,4,70% of present investment intensity,investment index,0.70


---

## Step 5 — The ten-year cash flow, line by line

Now the actual calculation. For each of ten years the model asks: *how much more liquidity does
this company have because it holds this patent?* Six components make up the answer — four that
add, two that subtract:

| Component | Sign | What it means |
|---|---|---|
| **Revenue** | + | the extra turnover the technology brings (question C6), growing with the market (C2) |
| **Regained revenue** | + | the turnover that would have been *lost* without the patent (question D1) |
| **Efficiency** | + | savings if production gets cheaper with the technology (D3) |
| **Investment reduction** | + | equipment you no longer need to buy if the technology is less capital-hungry (D4) |
| **Costs** | − | production costs on the new revenue, plus development costs until market entry (D2, D3) |
| **Investments** | − | the one-time equipment investment needed to produce it (D4) |

Add them up and you get the **liquidity** of that year. Discount each year back to today at the
company's discount rate, sum the ten — and that is the **Net Present Value**.

One reading tip for the table: the numbers are expressed as **percent of business turnover**,
the way the EPO workbook does it. Only the last column converts to Euro.

In [9]:
flow = kit.cash_flow(company, oek)

table = pd.DataFrame([{
    "Year": r.year,
    "On market": f"{r.active_fraction:.0%}",
    "Revenue": r.revenue,
    "Regained": r.regained_revenue,
    "Efficiency": r.efficiency,
    "Inv. reduction": r.investment_reduction,
    "Costs": -r.costs,
    "Investments": -r.investments,
    "Liquidity": r.liquidity,
    "Discounted (€)": r.discounted,
} for r in flow])

display(table.round(2))
print(f"\nNet Present Value:  € {sum(r.discounted for r in flow):,.0f}")

,Year,On market,Revenue,Regained,Efficiency,Inv. reduction,Costs,Investments,Liquidity,Discounted (€)
0,1,0%,0.00,0.00,0.00,0.00,-3.20,-0.00,-3.20,-42857.14
1,2,100%,2.02,8.75,5.25,2.94,-1.28,-0.62,17.05,203878.58
2,3,100%,2.35,9.45,5.67,0.00,-1.50,-0.00,15.97,170493.66
3,4,100%,2.74,10.20,6.12,0.00,-1.75,-0.00,17.32,165106.51
4,5,100%,3.20,11.02,6.61,0.00,-2.04,-0.00,18.79,159940.83
5,6,100%,3.73,11.90,7.14,0.00,-2.38,-0.00,20.39,154989.93
6,7,100%,4.35,12.85,7.71,0.00,-2.77,-0.00,22.14,150247.38
7,8,100%,5.08,13.88,8.33,0.00,-3.24,-0.00,24.05,145707.06
8,9,100%,5.92,14.99,9.00,0.00,-3.77,-0.00,26.13,141363.09
9,10,0%,0.00,0.00,0.00,0.00,-0.00,-0.00,0.00,0.00



Net Present Value:  € 1,248,870


The same table as a picture. Bars above the line add liquidity, bars below subtract it; the
black line is what remains in each year, before discounting.

Watch three things. **Years 1 and 2 are pure cost** — the technology is not on the market yet,
but development is being paid for, so liquidity is negative. **The equipment investment lands
once**, in the year of market entry, and never again. And **the whole thing stops when the
technology's life expectancy runs out** — here after four years on the market — whether or not
the patent is still in force. A patent with twelve years of term left protects a technology
that the model expects to be commercially irrelevant long before that.

In [10]:
components = [
    ("Revenue", [r.revenue for r in flow], kit.PALETTE["revenue"]),
    ("Regained revenue", [r.regained_revenue for r in flow], kit.PALETTE["regained"]),
    ("Efficiency", [r.efficiency for r in flow], kit.PALETTE["efficiency"]),
    ("Investment reduction", [r.investment_reduction for r in flow], kit.PALETTE["investment_reduction"]),
    ("Costs", [-r.costs for r in flow], kit.PALETTE["costs"]),
    ("Investments", [-r.investments for r in flow], kit.PALETTE["investments"]),
]
years = [r.year for r in flow]

fig = go.Figure()
for name, values, color in components:
    fig.add_trace(go.Bar(
        x=years, y=values, name=name,
        marker={"color": color, "line": {"width": 1, "color": kit.PALETTE["surface"]}},
        hovertemplate="%{y:.2f} % of turnover<extra>" + name + "</extra>",
    ))
fig.add_trace(go.Scatter(
    x=years, y=[r.liquidity for r in flow], name="Liquidity (net)",
    mode="lines+markers", line={"color": kit.PALETTE["ink"], "width": 2},
    marker={"size": 8}, hovertemplate="%{y:.2f} % of turnover<extra>Liquidity</extra>",
))

fig.update_layout(
    **kit.CHART_LAYOUT,
    barmode="relative", height=480, hovermode="x unified",
    title="What the patent adds to the company's liquidity, year by year",
)
fig.update_xaxes(title="Year", dtick=1)
fig.update_yaxes(title="% of business turnover")
fig.show()

---

## Step 6 — Two things everybody gets wrong

If you ever re-implement this model — and the point of this notebook is that you now could —
these are the two places it will break. Both are visible in the table above.

**1 · Entry and exit years are fractional.** "Two and a half years to market" means the third
year is only half a year on the market, and earns half the revenue. The same happens at the
end: a technology with a four-year life expectancy that arrived mid-year leaves mid-year. Treat
the years as whole and every number comes out too high.

**2 · Investments happen once, not every year.** The equipment investment and the investment
reduction are one-time events in the year the technology reaches the market. Spreading them
across all ten years is the classic mistake, and it is a large one — it turns a positive
valuation into a negative one.

In [11]:
demo = pd.DataFrame([{
    "Year": y,
    "T = 0.5 years": kit._active_fraction(y, 0.5, 4),
    "T = 2.5 years": kit._active_fraction(y, 2.5, 4),
    "T = 5 years": kit._active_fraction(y, 5, 4),
} for y in range(1, 11)])

print("How much of each year the technology is actually on the market")
print("(life expectancy 4 years in all three columns):\n")
display(demo.set_index("Year"))

launch = [r.year for r in flow if r.investments or r.investment_reduction]
print(f"Years carrying a one-time investment effect in our example: {launch}")

How much of each year the technology is actually on the market
(life expectancy 4 years in all three columns):



,T = 0.5 years,T = 2.5 years,T = 5 years
Year,,,
1,0.5,0.0,0.0
2,1.0,0.0,0.0
3,1.0,0.5,0.0
4,1.0,1.0,0.0
5,0.5,1.0,0.0
6,0.0,1.0,1.0
7,0.0,0.5,1.0
8,0.0,0.0,1.0
9,0.0,0.0,1.0


Years carrying a one-time investment effect in our example: [2]


---

## Step 7 — The acceptance test: does our engine equal the EPO's?

Everything above is *our* implementation. It looks reasonable, the chart looks plausible, the
numbers have the right order of magnitude — and none of that is evidence.

The EPO workbook ships with **three built-in test patents** and its own computed Net Present
Value for each. That is a source of truth we did not write. Our engine has to reproduce all
three, to the cent, before anything it says is worth hearing.

Two of the three are deliberately awkward: one lands just above zero, one below it. A model
that gets the easy case right and the sign wrong on the hard one is worse than no model.

This is the transferable lesson of the notebook, and it has nothing to do with patents: **check
against the source of truth, not against your own previous output.** This exact check, run
against the engine in the IPScore reference, once caught a real off-by-one bug that two demo cases had
happily hidden.

In [12]:
results = pd.DataFrame(kit.verify())
results["difference"] = results["difference"].map(lambda d: f"{d:+.2e}")
display(results.style.format({"computed": "{:,.4f}", "expected": "{:,.4f}"}))

assert all(r["passed"] for r in kit.verify()), "engine does not reproduce the EPO workbook"
print("\n✓ All three EPO test patents reproduced. The engine is trustworthy.")

,name,computed,expected,difference,passed
0,Patent 1,"329,059.4284","329,059.4284",+5.82e-11,True
1,Patent 2,"4,361.2849","4,361.2849",-1.82e-12,True
2,Patent 3,"-4,686.3598","-4,686.3598",-1.00e-11,True



✓ All three EPO test patents reproduced. The engine is trustworthy.


---

## What you now have — and what is still missing

You have a working, verified implementation of the EPO IPScore model: 40 questions, a profile,
and a Net Present Value that matches the EPO's own workbook to the cent.

And you have the model's honest weakness in plain sight. **Every one of those 40 answers came
from a person.** Nothing in IPScore checked a single fact about the patent — not whether it is
granted, not how long it runs, not where it is in force. The engine is exact; its inputs are
opinions. An exact machine fed opinions produces a very confident-looking opinion.

A PATLIB, however, has PATSTAT. Roughly six of these forty questions are matters of record, and
a few more can at least be given context. That is what the next notebook does: take a real
patent, answer what can be answered from data, mark every answer with where it came from — and
leave the rest visibly blank.

**Next: `2_evidence_from_patstat.ipynb`** — the only notebook in this module that needs EPO TIP.